# Hito 1 Baseline - F1 Race Strategy Advisor

This notebook implements the locked Hito 1 target (`is_top10`) and temporal split: train 2019-2021, calibration 2022, test 2023-2024. Strategy fields are treated as user-controlled scenario inputs for what-if comparison, not as normal pre-race signals.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_SEED = 414
DATA_PATH = Path('data/f1_strategy_race_level.csv')
pd.set_option('display.max_columns', 120)

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())
assert {'season', 'is_top10', 'n_stops', 'compound_sequence'}.issubset(df.columns)

## 2. Locked Temporal Split

In [ ]:
train = df[df['season'].between(2019, 2021)].copy()
cal = df[df['season'].eq(2022)].copy()
test = df[df['season'].between(2023, 2024)].copy()

print('train seasons:', sorted(train['season'].unique()), train.shape)
print('calibration seasons:', sorted(cal['season'].unique()), cal.shape)
print('test seasons:', sorted(test['season'].unique()), test.shape)

assert set(train['season'].unique()) == {2019, 2020, 2021}
assert set(cal['season'].unique()) == {2022}
assert set(test['season'].unique()) == {2023, 2024}
assert len(train) + len(cal) + len(test) == len(df)

## 3. Leakage Audit

In [ ]:
pre_race_features = [
    'grid_position', 'qualifying_position', 'front_grid_start', 'back_grid_start',
    'driver_recent_dnf_rate', 'driver_dnf_any_last3', 'team_season_dnf_rate_to_date',
    'driver_experience_races', 'constructor_recent_dnf_rate_10', 'circuit_prior_dnf_rate',
    'driver_avg_grid_last5', 'constructor_avg_grid_last10', 'grid_vs_driver_avg_last5',
    'grid_vs_constructor_avg_last10', 'practice_sessions_with_timed_laps',
    'practice_timed_laps', 'practice_avg_gap_to_fastest_sec', 'practice_min_gap_to_fastest_sec',
    'practice_total_laps', 'practice_deleted_laps', 'practice_yellow_lap_count',
    'practice_red_flag_lap_count', 'qualifying_timed_laps', 'qualifying_gap_to_pole_sec',
    'qualifying_deleted_laps', 'qualifying_yellow_lap_count', 'qualifying_red_flag_lap_count',
    'practice_sessions_available', 'qualifying_sessions_available', 'practice_completion_rate',
    'has_qualifying_time', 'has_practice_time'
]
scenario_inputs = ['n_stops', 'compound_sequence', 'avg_stint_tyre_life', 'max_stint_tyre_life', 'pit_loss_total']
categorical_features = ['constructor_name', 'circuit_type', 'compound_sequence']
audit_columns = ['season', 'round', 'race_name', 'driver_name', 'position', 'points', 'status', 'will_dnf', 'safety_car_periods', 'stint_lengths', 'data_source_note']

audit = pd.DataFrame([
    {'column_group': 'pre-race predictors', 'columns': ', '.join(pre_race_features), 'model_use': 'Allowed as information available before the race.'},
    {'column_group': 'scenario inputs', 'columns': ', '.join(scenario_inputs), 'model_use': 'Allowed only because the product compares user-defined strategy scenarios.'},
    {'column_group': 'audit / outcome columns', 'columns': ', '.join(audit_columns), 'model_use': 'Not used as predictors for fitting or selection.'},
])
display(audit)
for col in ['position', 'points', 'status', 'will_dnf', 'safety_car_periods', 'stint_lengths']:
    assert col not in pre_race_features + scenario_inputs + categorical_features

## 4. F1-Defendable Grid-Rule Baseline

In [ ]:
def grid_rule_proba(frame):
    grid = frame['grid_position'].fillna(20)
    return np.select(
        [grid <= 5, grid <= 10, grid <= 15],
        [0.88, 0.62, 0.24],
        default=0.08,
    ).astype(float)

grid_test_proba = grid_rule_proba(test)
grid_metrics = {
    'model': 'Grid-rule heuristic',
    'brier': brier_score_loss(test['is_top10'], grid_test_proba),
    'log_loss': log_loss(test['is_top10'], grid_test_proba, labels=[0, 1]),
    'roc_auc': roc_auc_score(test['is_top10'], grid_test_proba),
}
grid_metrics

## 5. Simple Calibrated Baseline Model

In [ ]:
numeric_features = pre_race_features + ['n_stops', 'avg_stint_tyre_life', 'max_stint_tyre_life', 'pit_loss_total']
feature_cols = numeric_features + categorical_features

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=5)),
])
preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features),
])
model = GradientBoostingClassifier(
    learning_rate=0.04,
    n_estimators=140,
    max_depth=2,
    min_samples_leaf=20,
    random_state=RANDOM_SEED,
)
clf = Pipeline([('preprocess', preprocess), ('model', model)])

X_train, y_train = train[feature_cols], train['is_top10']
X_cal, y_cal = cal[feature_cols], cal['is_top10']
X_test, y_test = test[feature_cols], test['is_top10']

clf.fit(X_train, y_train)
cal_raw = clf.predict_proba(X_cal)[:, 1]
test_raw = clf.predict_proba(X_test)[:, 1]

iso = IsotonicRegression(out_of_bounds='clip', y_min=0.001, y_max=0.999)
iso.fit(cal_raw, y_cal)
test_proba = iso.predict(test_raw)

model_metrics = {
    'model': 'Calibrated GB baseline',
    'brier': brier_score_loss(y_test, test_proba),
    'log_loss': log_loss(y_test, test_proba, labels=[0, 1]),
    'roc_auc': roc_auc_score(y_test, test_proba),
}
model_metrics

## 6. Test Metrics vs Reference Floors

In [ ]:
results = pd.DataFrame([grid_metrics, model_metrics])
results['beats_grid_rule_brier_0.208'] = results['brier'] < 0.208
results['beats_docent_brier_0.132'] = results['brier'] < 0.132
results['beats_docent_auc_0.892'] = results['roc_auc'] > 0.892
display(results)

best = results.sort_values('brier').iloc[0]
print(f"Best Hito 1 baseline by Brier: {best['model']} | Brier={best['brier']:.3f}, LogLoss={best['log_loss']:.3f}, AUC={best['roc_auc']:.3f}")

## 7. Calibration Curve on Test Set

In [ ]:
calibration_df = pd.DataFrame({'y': y_test, 'p': test_proba})
calibration_df['bin'] = pd.qcut(calibration_df['p'], q=10, duplicates='drop')
curve = calibration_df.groupby('bin', observed=True).agg(
    mean_pred=('p', 'mean'),
    observed_rate=('y', 'mean'),
    n=('y', 'size'),
).reset_index(drop=True)
display(curve)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='perfect calibration')
ax.plot(curve['mean_pred'], curve['observed_rate'], marker='o', label='calibrated baseline')
for _, row in curve.iterrows():
    ax.annotate(int(row['n']), (row['mean_pred'], row['observed_rate']), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('Mean predicted P(top10)')
ax.set_ylabel('Observed top10 rate')
ax.set_title('Test calibration curve, 2023-2024')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
plt.show()

## 8. Concrete What-If Example

In [ ]:
example = test[(test['season'].eq(2024)) & (test['race_name'].str.contains('Monaco', case=False, na=False)) & (test['driver_name'].eq('Leclerc'))].head(1).copy()
if len(example) == 0:
    example = test.head(1).copy()
scenario_a = example.copy()
scenario_b = example.copy()
scenario_a[['n_stops', 'compound_sequence', 'avg_stint_tyre_life', 'max_stint_tyre_life', 'pit_loss_total']] = [1, 'MEDIUM-HARD', 34, 44, 22.5]
scenario_b[['n_stops', 'compound_sequence', 'avg_stint_tyre_life', 'max_stint_tyre_life', 'pit_loss_total']] = [2, 'MEDIUM-MEDIUM-HARD', 23, 30, 45.0]
scenario_probs = iso.predict(clf.predict_proba(pd.concat([scenario_a, scenario_b])[feature_cols])[:, 1])
pd.DataFrame({
    'scenario': ['one-stop M-H', 'two-stop M-M-H'],
    'driver': [example['driver_name'].iloc[0]] * 2,
    'race': [example['race_name'].iloc[0]] * 2,
    'predicted_P_top10': scenario_probs,
})